In [1]:
import pandas as pd
import os
from lsst.summit.utils import (
    ConsDbClient,
    getBandpassSeeingCorrection,
    getAirmassSeeingCorrection,
    makeDefaultButler
)
from sqlalchemy import create_engine
import numpy as np
from lsst.obs.lsst import LsstCam
from lsst.ts.ofc import BendModeToForce, OFCData, StateEstimator
from lsst.ts.wep.utils import convertZernikesToPsfWidth


In [2]:
# Define ConsDbClient with specific URL
URL = "http://consdb-pq.consdb:8080/consdb"  
os.environ["no_proxy"] += ",.consdb"
cdb_client = ConsDbClient(URL)

ofc_data = OFCData("lsst")
ofc_data.controller["truncation_index"] = 6

m2_hexapod = np.ones(5, dtype=bool)
cam_hexapod = np.ones(5, dtype=bool)
m1m3_bending = np.zeros(20, dtype=bool)
m2_bending = np.zeros(20, dtype=bool)
m1m3_bending[:5] = True
m2_bending[:5] = True
ofc_data.comp_dof_idx = dict(
    m2HexPos=m2_hexapod,
    camHexPos=cam_hexapod,
    M1M3Bend=m1m3_bending,
    M2Bend=m2_bending,
)
state_estimator = StateEstimator(ofc_data)

m2_bmf = BendModeToForce("M2", ofc_data)
m1m3_bmf = BendModeToForce("M1M3", ofc_data)

det_order = (191, 195, 199, 203)
camera = LsstCam().getCamera()
detector_names = [
    camera.get(det_id).getName() for det_id in det_order
]

In [3]:
query_table = """
        SELECT
            e.air_temp AS air_temp,
            e.airmass AS airmass,
            e.dimm_seeing AS dimm,
            e.altitude AS elevation,
            e.azimuth AS azimuth,
            e.exposure_id AS exposure_id,
            q.visit_id AS visit_id,
            e.physical_filter as band,
            e.day_obs AS day,
            e.exp_midpt AS time,
            e.dimm_seeing AS seeing,
            e.humidity AS humidity,
            e.pressure AS pressure,
            e.seq_num AS seq,
            e.sky_rotation AS sky_rotation,
            e.wind_dir AS wind_dir,
            e.wind_speed AS wind_speed,
            q.psf_sigma_median AS psf_fwhm,
            q.aos_fwhm,
            ccdvisit1_quicklook.z4,
            ccdvisit1_quicklook.z5,
            ccdvisit1_quicklook.z6,
            ccdvisit1_quicklook.z7,
            ccdvisit1_quicklook.z8,
            ccdvisit1_quicklook.z9,
            ccdvisit1_quicklook.z10,
            ccdvisit1_quicklook.z11,
            ccdvisit1_quicklook.z12,
            ccdvisit1_quicklook.z13,
            ccdvisit1_quicklook.z14,
            ccdvisit1_quicklook.z15,
            ccdvisit1_quicklook.z16,
            ccdvisit1_quicklook.z17,
            ccdvisit1_quicklook.z18,
            ccdvisit1_quicklook.z19,
            ccdvisit1_quicklook.z20,
            ccdvisit1_quicklook.z21,
            ccdvisit1_quicklook.z22,
            ccdvisit1_quicklook.z23,
            ccdvisit1_quicklook.z24,
            ccdvisit1_quicklook.z25,
            ccdvisit1_quicklook.z26,
            ccdvisit1_quicklook.z27,
            ccdvisit1_quicklook.z28,
            ccdvisit1.detector as detector,
            efd.mt_pointing_mount_position_rotator_mean AS rotation_angle,
            efd.mt_salindex110_sonic_temperature_mean AS sonic_temperature,
            efd.mt_salindex110_sonic_temperature_stddev_mean AS sonic_temperature_std,
            efd.mt_salindex1_temperature_0_mean AS cam_hex_temp_0,
            efd.mt_salindex1_temperature_1_mean AS cam_hex_temp_1,
            efd.mt_salindex1_temperature_2_mean AS cam_hex_temp_2,
            efd.mt_salindex1_temperature_3_mean AS cam_hex_temp_3,
            efd.mt_salindex1_temperature_4_mean AS cam_hex_temp_4,
            efd.mt_salindex1_temperature_5_mean AS cam_hex_temp_5,
            efd.mt_salindex1_temperature_6_mean AS cam_hex_temp_6,
            efd.mt_salindex1_temperature_7_mean AS cam_hex_temp_7,
            efd.mt_salindex301_temperature_0_mean AS outside_temp_0,
            efd.mt_salindex301_temperature_1_mean AS outside_temp_1,
            efd.mt_salindex301_temperature_2_mean AS outside_temp_2,
            efd.mt_salindex301_temperature_3_mean AS outside_temp_3,
            efd.mt_salindex301_temperature_4_mean AS outside_temp_4,
            efd.mt_salindex301_temperature_5_mean AS outside_temp_5,
            efd.mt_salindex301_temperature_6_mean AS outside_temp_6,
            efd.mt_salindex301_temperature_7_mean AS outside_temp_7
        FROM 
            cdb_lsstcam.exposure AS e,
            cdb_lsstcam.visit1_quicklook AS q,
            efd_lsstcam.exposure_efd AS efd,
            cdb_lsstcam.ccdvisit1_quicklook AS ccdvisit1_quicklook,
            cdb_lsstcam.ccdvisit1 AS ccdvisit1
        WHERE 
            ccdvisit1.detector IN (191, 192, 195, 196, 199, 200, 203, 204)
            AND ccdvisit1.ccdvisit_id = ccdvisit1_quicklook.ccdvisit_id
            AND ccdvisit1.visit_id = q.visit_id
            AND ccdvisit1.visit_id = e.exposure_id
            AND ccdvisit1.visit_id = efd.exposure_id
            AND (e.img_type = 'science' or e.img_type = 'acq')
            AND e.day_obs > 20250528
            AND e.airmass > 0
            AND e.band != 'none'
        
        --order-by e.seq_num
"""
result_df = cdb_client.query(query_table).to_pandas()
zernike_columns = [f"z{i}" for i in range(4, 29)]
result_df["zernikes"] = result_df[zernike_columns].apply(
    lambda row: np.array(row.fillna(0.0).values, dtype=float), axis=1
)

/tmp/ipykernel_20657/1737501411.py:89: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda row: np.array(row.fillna(0.0).values, dtype=float), axis=1


In [4]:
result_df.columns

Index(['air_temp', 'airmass', 'dimm', 'elevation', 'azimuth', 'exposure_id',
       'visit_id', 'band', 'day', 'time', 'seeing', 'humidity', 'pressure',
       'seq', 'sky_rotation', 'wind_dir', 'wind_speed', 'psf_fwhm', 'aos_fwhm',
       'z4', 'z5', 'z6', 'z7', 'z8', 'z9', 'z10', 'z11', 'z12', 'z13', 'z14',
       'z15', 'z16', 'z17', 'z18', 'z19', 'z20', 'z21', 'z22', 'z23', 'z24',
       'z25', 'z26', 'z27', 'z28', 'detector', 'rotation_angle',
       'sonic_temperature', 'sonic_temperature_std', 'cam_hex_temp_0',
       'cam_hex_temp_1', 'cam_hex_temp_2', 'cam_hex_temp_3', 'cam_hex_temp_4',
       'cam_hex_temp_5', 'cam_hex_temp_6', 'cam_hex_temp_7', 'outside_temp_0',
       'outside_temp_1', 'outside_temp_2', 'outside_temp_3', 'outside_temp_4',
       'outside_temp_5', 'outside_temp_6', 'outside_temp_7', 'zernikes'],
      dtype='object')

In [31]:
# At this point we have queried the classic consdb which is also available at the summit
# now let's dig through the most interesting one, the efd_lsstcam
# you may consider adding more properties, here is a few that I am interested in
# Query unpivoted data based on the previous result 
exposure_ids = ', '.join(map(str, result_df["exposure_id"].values))

query_unpivoted = f"""
SELECT
    exposure_id,
    property,
    field,
    value
FROM
    efd_lsstcam.exposure_efd_unpivoted
WHERE
    exposure_id IN ({exposure_ids})
    AND (property = 'mt_m2_axial_force_lut_gravity_mean'
        OR property = 'mt_m2_axial_force_lut_temperature_mean'
        OR property = 'mt_m1m3_applied_elevation_forces_mean'
        OR property = 'mt_logevent_aggregated_dof'
        OR property = 'mt_pointing_mount_position_rotator_mean'
    );
"""

rename_dict = {'mt_m2_axial_force_lut_gravity_mean':'m2',
               'mt_m2_axial_force_lut_temperature_mean':'m2',
               'mt_m1m3_applied_azimuth_forces_mean': 'azimuth',
               'mt_m1m3_applied_thermal_forces_mean': 'thermal_forces',
               'mt_m1m3_applied_elevation_forces_mean': 'elevation',
               'mt_logevent_aggregated_dof':'',
              }
result_unp_df = cdb_client.query(query_unpivoted).to_pandas()

result_unp_df['property'] = result_unp_df['property'].replace(rename_dict)

result_unp_df['property_field'] = result_unp_df['property'] + '_' + result_unp_df['field']

result_unp_df = result_unp_df.pivot(columns='property_field', values='value', index='exposure_id').reset_index()

# Merge tabular and unpivoted dataframes
result = pd.merge(result_df, result_unp_df, on='exposure_id', how='inner')

gravity_cols = [f"m2_lutGravity{i}" for i in range(72)]
temp_cols = [f"m2_lutTemperature{i}" for i in range(72)]

result["m2_lut_gravity"] = result[gravity_cols].values.tolist()
result["m2_lut_temp"] = result[temp_cols].values.tolist()
gravity_array = result[gravity_cols].values.astype(float)
temp_array = result[temp_cols].values.astype(float)

# Element-wise sum across each row
result["m2_lut"] = (gravity_array + temp_array).tolist()
result["m2_bmf"] = result["m2_lut"].apply(lambda x: m2_bmf.bending_mode(np.array(x)))

result.drop(columns=gravity_cols, inplace=True)
result.drop(columns=temp_cols, inplace=True)

force_types = ['elevation']
cols_list = [[f"{ft}_zForces{i}" for i in range(156)] for ft in force_types]
all_cols = [col for cols in cols_list for col in cols]

result["m1m3_lut"] = (result[cols_list[0]].values).tolist()
result["m1m3_bmf"] = result["m1m3_lut"].apply(lambda x: m1m3_bmf.bending_mode(np.array(x)))
result.drop(columns=all_cols, inplace=True)

aggregated_dof_cols = [f"_aggregatedDoF{i}" for i in range(50)]
values = result[aggregated_dof_cols].values.tolist()
result["dof_state"] = result[aggregated_dof_cols].values.tolist()
result.drop(columns=aggregated_dof_cols, inplace=True)

temp_cols = [f"cam_hex_temp_{i}" for i in range(8)]
result["cam_hex_temp_avg"] = result[temp_cols].mean(axis=1)
result.drop(columns=temp_cols, inplace=True)

outside_temp_cols = [f"outside_temp_{i}" for i in range(8)]
result["outside_temp_avg"] = result[outside_temp_cols].mean(axis=1)
result.drop(columns=outside_temp_cols, inplace=True)

result

,air_temp,airmass,dimm,elevation,azimuth,exposure_id,visit_id,band,day,time,...,_visitDoF49,m2_lut_gravity,m2_lut_temp,m2_lut,m2_bmf,m1m3_lut,m1m3_bmf,dof_state,cam_hex_temp_avg,outside_temp_avg
0,3.150,1.272032,None,51.682424,246.935815,2025060600085,2025060600085,i_39,20250606,2025-06-07T00:11:13.617000,...,NaN,"[72.31517741935484, 195.51517741935484, 123.46...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[72.31517741935484, 195.51517741935484, 123.46...","[-1.0935428114307983, 13.023454520975532, -1.7...","[590.8649222488802, 663.8115417720136, 782.662...","[0.0528401151287099, 0.2462285053189437, -1.32...","[1250.319727143461, 68.50031924420232, -567.19...",10.144261,3.126563
1,3.150,1.272032,None,51.682424,246.935815,2025060600085,2025060600085,i_39,20250606,2025-06-07T00:11:13.617000,...,NaN,"[72.31517741935484, 195.51517741935484, 123.46...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[72.31517741935484, 195.51517741935484, 123.46...","[-1.0935428114307983, 13.023454520975532, -1.7...","[590.8649222488802, 663.8115417720136, 782.662...","[0.0528401151287099, 0.2462285053189437, -1.32...","[1250.319727143461, 68.50031924420232, -567.19...",10.144261,3.126563
2,3.150,1.272032,None,51.682424,246.935815,2025060600085,2025060600085,i_39,20250606,2025-06-07T00:11:13.617000,...,NaN,"[72.31517741935484, 195.51517741935484, 123.46...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[72.31517741935484, 195.51517741935484, 123.46...","[-1.0935428114307983, 13.023454520975532, -1.7...","[590.8649222488802, 663.8115417720136, 782.662...","[0.0528401151287099, 0.2462285053189437, -1.32...","[1250.319727143461, 68.50031924420232, -567.19...",10.144261,3.126563
3,3.150,1.272032,None,51.682424,246.935815,2025060600085,2025060600085,i_39,20250606,2025-06-07T00:11:13.617000,...,NaN,"[72.31517741935484, 195.51517741935484, 123.46...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[72.31517741935484, 195.51517741935484, 123.46...","[-1.0935428114307983, 13.023454520975532, -1.7...","[590.8649222488802, 663.8115417720136, 782.662...","[0.0528401151287099, 0.2462285053189437, -1.32...","[1250.319727143461, 68.50031924420232, -567.19...",10.144261,3.126563
4,5.375,1.354218,None,47.484063,334.382491,2025060700512,2025060700512,g_6,20250607,2025-06-08T08:36:28.055000,...,NaN,"[54.38962843295638, 185.70705977382877, 112.71...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[54.38962843295638, 185.70705977382877, 112.71...","[-1.1036969948700897, 13.025002374234607, -1.7...","[554.8988148191755, 623.170153463717, 735.1143...","[0.061464604917704335, 0.23787476806969132, -1...","[431.2187219144393, 68.50015222399779, -567.19...",9.718409,5.359375
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19231,6.650,1.402128,None,45.331914,261.581466,2025063000686,2025063000686,i_39,20250630,2025-07-01T09:23:36.060000,...,NaN,"[44.973090614886736, 180.46765372168286, 107.0...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[44.973090614886736, 180.46765372168286, 107.0...","[-1.1091344771029519, 13.026363532510713, -1.7...","[535.5151024189934, 601.2764950211322, 709.488...","[0.06568614857236499, 0.23312922001684766, -1....","[2469.6773622167416, -69.42905670765604, 567.3...",11.655192,6.65625
19232,7.000,1.214074,None,55.314204,263.345046,2025063000693,2025063000693,r_57,20250630,2025-07-01T09:30:26.080000,...,NaN,"[86.80200323101778, 203.1860743134087, 132.073...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[86.80200323101778, 203.1860743134087, 132.073...","[-1.0856164878969325, 13.023493482454782, -1.7...","[619.3414656099957, 696.0124364641256, 820.309...","[0.04529189970830938, 0.2523514799931361, -1.3...","[2154.3776040371167, -69.43171066517482, 567.3...",11.596664,6.9625
19233,7.000,1.214074,None,55.314204,263.345046,2025063000693,2025063000693,r_57,20250630,2025-07-01T09:30:26.080000,...,NaN,"[86.80200323101778, 203.1860743134087, 132.073...","[0.0, 0.0, 0.0, 

In [70]:
import pandas as pd
import os
from lsst.summit.utils import (
    ConsDbClient,
    getBandpassSeeingCorrection,
    getAirmassSeeingCorrection,
    makeDefaultButler
)

# Define ConsDbClient with specific URL
URL = "http://consdb-pq.consdb:8080/consdb"  
os.environ["no_proxy"] += ",.consdb"
cdb_client = ConsDbClient(URL)

# Query tabular data from ConsDbClient and convert to DataFrame
query_table = """
        SELECT
            e.air_temp AS air_temp,
            e.airmass AS airmass,
            e.dimm_seeing AS dimm,
            e.altitude AS elevation,
            e.azimuth AS azimuth,
            e.exposure_id AS exposure_id,
            e.physical_filter as band,
            e.day_obs AS day,
            e.exp_midpt AS time,
            e.dimm_seeing AS seeing,
            e.humidity AS humidity,
            e.pressure AS pressure,
            e.seq_num AS seq,
            e.sky_rotation AS sky_rotation,
            e.wind_dir AS wind_dir,
            e.wind_speed AS wind_speed,
            q.sky_noise_median AS sky_noise,
            q.sky_noise_max AS sky_noise_max,
            q.sky_noise_min AS sky_noise_min,
            q.sky_bg_median AS sky_bg,
            q.sky_bg_max AS sky_bg_max,
            q.sky_bg_min AS sky_bg_min,
            q.psf_sigma_median AS psf_fwhm,
            q.psf_sigma_min AS psf_fwhm_min,
            q.psf_sigma_max AS psf_fwhm_max,
            q.psf_ixx_median AS psf_ixx_median,
            q.psf_ixx_max AS psf_ixx_max,
            q.psf_ixx_min AS psf_ixx_min,
            q.psf_iyy_median AS psf_iyy_median,
            q.psf_iyy_max AS psf_iyy_max,
            q.psf_iyy_min AS psf_iyy_min,
            q.psf_ixy_median AS psf_ixy_median,
            q.psf_ixy_max AS psf_ixy_max,
            q.psf_ixy_min AS psf_ixy_min,
            q.psf_area_max AS psf_area_max,
            q.psf_area_median AS psf_area_median,
            q.psf_area_min AS psf_area_min
        FROM 
            cdb_lsstcam.exposure AS e
        JOIN 
            cdb_lsstcam.visit1_quicklook AS q
            ON e.exposure_id = q.visit_id
        WHERE 
            (e.img_type = 'OBJECT' or e.img_type = 'ACQ') AND
            e.day_obs > 20250501 AND
            e.airmass > 0 AND
            e.band != 'none'
        
        --order-by e.seq_num
"""
result_df = cdb_client.query(query_table).to_pandas()

# Query unpivoted data based on the previous result 
exposure_ids = ', '.join(map(str, result_df["exposure_id"].values))
query_unpivoted = f"""
SELECT
    exposure_id,
    property,
    field,
    value
FROM
    efd_lsstcam.exposure_efd_unpivoted
WHERE
    exposure_id IN ({exposure_ids})
    AND property = 'mt_m2_axial_force_lut_temperature_mean'
"""


result_unp_df = cdb_client.query(query_unpivoted).to_pandas()
print(result_unp_df)
result_unp_df['property_field'] = result_unp_df['property'] + '_' + result_unp_df['field']

result_unp_df = result_unp_df.pivot(columns='property_field', values='value', index='exposure_id').reset_index()

# Merge tabular and unpivoted dataframes
result = pd.merge(result_df, result_unp_df, on='exposure_id', how='inner')

Empty DataFrame
Columns: [exposure_id, property, field, value]
Index: []


UFuncTypeError: ufunc 'add' did not contain a loop with signature matching types (dtype('float64'), dtype('<U1')) -> None